In [ ]:
%%bash
rm -rf ~/.cache/huggingface/hub
rm -rf ~/.cache/huggingface/transformers

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
# !pip install --index-url https://download.pytorch.org/whl/cu124 torch torchvision torchaudio
!pip install --index-url https://download.pytorch.org/whl/cu126 torch torchvision torchaudio
# !pip install torch torchvision torchaudio

In [ ]:
import os
import json
import time

import pandas as pd
from tqdm.auto import tqdm
from io import StringIO
import os

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import redshift_connector
import yaml
import boto3

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("./utils").resolve()))

from asksa_entity import (
    get_topics_df, ExtractConfig,
    build_resources, extract_raw_matches, drop_contraction_artifacts, build_group_payloads,
    load_llm_pipeline, run_llm_batched, build_validated_df, build_llm_prompt
)

In [ ]:
print("torch:", torch.__version__)
print("torch cuda version:", torch.version.cuda)   # якщо None -> CPU build
print("is_available:", torch.cuda.is_available())

In [ ]:
with open("./variables.yaml", "r") as yamlfile:
    cfg = yaml.load(yamlfile, Loader=yaml.FullLoader)

REGION_NAME = cfg['REGION_NAME']
SECRET_NAME = cfg['SECRET_NAME']


BUCKET_NAME = cfg['BUCKET_NAME']
S3_PROJECT_DIR = cfg['S3_PROJECT_DIR']
UPDATE_TABLE_DIR = cfg['UPDATE_TABLE_DIR']

HOST = cfg['HOST']
PORT = cfg['PORT']
DATABASE = cfg['DATABASE']

os.environ["HF_TOKEN"] = cfg['HFT']

In [ ]:
session = boto3.session.Session()
secrets_manager_client = session.client(
    service_name = 'secretsmanager',
    region_name = REGION_NAME
)

In [ ]:
secret_name = SECRET_NAME
response = secrets_manager_client.get_secret_value(SecretId=secret_name)

if 'SecretString' in response:
    secret = response['SecretString']
else:
    secret = response['SecretBinary']

In [ ]:
parsed_secret = json.loads(secret)
redshift_username = parsed_secret['username']
redshift_password = parsed_secret['password']

In [ ]:
obj = boto3.client('s3')

In [ ]:
querie_get_ticker_data = './queries/get_tickers.sql'
fd = open(querie_get_ticker_data, 'r')
get_data = fd.read()
fd.close()

In [ ]:
querie_get_user_chat = './queries/get_all_user_chats.sql'
fd = open(querie_get_user_chat, 'r')
get_chat_data = fd.read()
fd.close()

In [ ]:
conn = redshift_connector.connect(
    host = cfg['HOST'],
    port = cfg['PORT'],
    database = cfg['DATABASE'],
    user = redshift_username,
    password = redshift_password
 )
cur = conn.cursor()

In [ ]:
cur.execute(get_data)
ticker_data_df = cur.fetch_dataframe()
ticker_data_df.head()

In [ ]:
cur.execute(get_chat_data)
messages_df = cur.fetch_dataframe()
messages_df.head()

In [ ]:
conn.close()

In [ ]:
topics_df = get_topics_df()
print("topics:", len(topics_df))
# topics_df.head()

In [ ]:
tickers_df = ticker_data_df[['ticker_id', 'company_name']].copy()
secind_df = ticker_data_df[['sector_name', 'industry_name']].drop_duplicates()

print("tickers:", len(tickers_df))
print("sector/industry rows:", len(secind_df))

In [ ]:
cfg = ExtractConfig(
    batch_size=5000,
    max_ngram_fallback=6
)
R = build_resources(tickers_df, secind_df, topics_df, cfg=cfg)

In [ ]:
raw_df = extract_raw_matches(messages_df, R)
raw_df = drop_contraction_artifacts(raw_df)

In [ ]:
groups_df = build_group_payloads(raw_df, max_candidates_per_msg=30)

In [ ]:
groups_df["prompt"] = groups_df.apply(
    lambda r: build_llm_prompt(r["msg"], r["payload"]),
    axis=1
)

In [ ]:
llm_pipe = load_llm_pipeline("google/gemma-2-9b-it", token=os.getenv("HF_TOKEN"))

In [ ]:
llm_outputs = run_llm_batched(
    llm_pipe,
    groups_df,
    batch_size=cfg.llm_batch_size,
    max_new_tokens=cfg.llm_max_new_tokens,
    repetition_penalty=cfg.llm_repetition_penalty
)

In [ ]:
validated_df = build_validated_df(groups_df, llm_outputs, debug=False)

In [ ]:
# def top_entities(df: pd.DataFrame, etype: str, n: int = 25) -> pd.DataFrame:
#     d = df[df.entity_type == etype].copy()
#     return (d.groupby(["entity_id"])
#              .size()
#              .reset_index(name="count")
#              .sort_values("count", ascending=False)
#              .head(n))

# display(top_entities(validated_df, "ticker", 25))
# display(top_entities(validated_df, "company", 25))
# display(top_entities(validated_df, "topic", 25))
# display(top_entities(validated_df, "sector", 25))
# display(top_entities(validated_df, "industry", 25))

In [ ]:
validated_df['msg_date'] = validated_df['ts'].dt.date

In [ ]:
csv_buf = StringIO()
validated_df.to_csv(csv_buf, header=True, index=False)
csv_buf.seek(0)
obj.put_object(Bucket = BUCKET_NAME, Body=csv_buf.getvalue(), Key = os.path.join(S3_PROJECT_DIR + UPDATE_TABLE_DIR, 'result_er.csv'))

In [ ]:
conn_info = {'host': HOST,
             'port': PORT,
             'user': redshift_username,
             'password': redshift_password,
             'database': DATABASE}

In [ ]:
# with redshift_connector.connect(**conn_info) as conn:
#     cur = conn.cursor()

#     drop_query = 'DROP TABLE IF EXISTS ds.production__ask_sa_message_entity_extraction'
#     cur.execute(drop_query)

#     create_table = 'CREATE TABLE ds.production__ask_sa_message_entity_extraction (ts timestamp, user_id int, chat_id int,\
#                                                                                   trace_id varchar(40), msg varchar(MAX),\
#                                                                                   entity_type varchar(12), entity_id varchar(100), msg_date date) DISTSTYLE EVEN SORTKEY(msg_date)'
#     cur.execute(create_table)

#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_message_entity_extraction TO analyst'

#     cur.execute(access)

#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_message_entity_extraction TO scientist'

#     cur.execute(access)
    
#     access = 'GRANT ALL PRIVILEGES ON TABLE ds.production__ask_sa_message_entity_extraction TO looker'
    
#     cur.execute(access)
    
#     conn.commit()

In [ ]:
with redshift_connector.connect(**conn_info) as conn:
     cur = conn.cursor()
     file_path = 's3://' +  BUCKET_NAME + '/' + S3_PROJECT_DIR + UPDATE_TABLE_DIR + 'result_er.csv'
     role_arn = 'arn:aws:iam::714212176917:role/redshift-access-to-seekingalpha-datascience-bucket'

     insert_query = """COPY ds.production__ask_sa_message_entity_extraction FROM '{}' IAM_ROLE '{}' csv IGNOREHEADER 1;""".format(file_path, role_arn)
     cur.execute(insert_query)
     conn.commit()

In [ ]:
with redshift_connector.connect(**conn_info) as conn:
    cur = conn.cursor()

    analyze_query = 'ANALYZE ds.production__ask_sa_message_entity_extraction'
    cur.execute(analyze_query)
    
    conn.commit()